# v1 commissioning (read-only)

One section per implementation stage (S0–S9). Cells only call functions from `twobunch_deploy`; each stage saves its records under `data/` (git-ignored, copied back by hand), and the result goes in the lab-notebook commissioning page.

Nothing here writes to the machine: `tests/test_read_only.py` fails on any write call in the package or in these cells.

## S0: environment

Reads no PV. Run on the host in the ACR prod Python environment, from the repo root (Jupyter started inside tmux).

Pass:
- `import_errors` is empty;
- `code_sha256` equals the value CI printed for this PR's head commit (the `pixi run code-hash` step);
- `git_commit` is that commit;
- `epics_env` shows the host's Channel Access settings (recorded; no target value).

In [ ]:
from twobunch_deploy import session

env = session.check_env()
env

In [ ]:
session.save_record("S0", "check_env", env)

## S1: camera frames and pulse IDs

Reads one PV, `CAMR:LI20:107:RAW:ArrayData` (the DTOTR2 camera image as the driver outputs it, the same arrays the DAQ saves; `Image:ArrayData` until S2): subscribes for 30 s, then disconnects. Needs the camera on, and beam for real images. Saves the summary as JSON and, next to it (`.npz`), every frame's pulse ID and timestamps plus 10 raw frames.

Pass:
- `frames` > 0 and `latency_s.first` under about 1 s: the camera is publishing, and the frame timestamps agree with this host's clock;
- `lengths` is a single value, the camera ROI's `SizeX` × `SizeY` (278,640 = 324 × 860 for the ROI read on 10-30 at 15:26; S2 checks the geometry);
- `min` ≥ 0 and `max` ≤ 4095 (12-bit camera); `dtype` recorded;
- `pulse_id_steps` almost all 36. A 0 next to a 72, or a 35 next to a 37, is the IOC stamping a frame with a neighbouring pulse ID (8 times in 33,779 DAQ frames); any other step is a finding;
- `capture_fraction` close to 1: the callback should get every frame. Lower means frames were lost between the IOC and the notebook.

In [ ]:
from twobunch_deploy import session

summary = session.capture_test(seconds=30)
summary

## S2: frame geometry

Reads the DTOTR2 camera readbacks once (`pvs.CAMERA_SETTINGS`: ROI, orientation, exposure, gain, pixel size, the driver's reverse and output size, and where the camera array comes from and its size), then subscribes to the camera for 3 s. From each frame it cuts the deployed analysis's hand crop (`screen.HAND_CROP`, 300 × 450; at the training orientation sensor columns 396–579 and rows 270–719, padded with zeros past row 184 as the DAQ classes' crop does). The crop is taken in sensor pixels, whatever ROI the camera is set to, and oriented with the live orientation readbacks. Needs beam on DTOTR2: run the S1 section again first, with beam, then this one. Saves the summary as JSON and, next to it, the pulse IDs, the newest raw frame and every frame's hand crop (`.npz`) and the figure (`.png`).

Pass:
- no `TimeoutError`: every readback answers, so the PV names are right;
- no `ValueError`: the camera array comes from the driver (port `PS107`), isn't reversed, and is the camera ROI's size (no binning); the camera's ROI contains the hand crop; each frame is `SizeX` × `SizeY` pixels;
- `trained_shape` is [300, 450];
- `differs` lists the readbacks that aren't the training values. The four ROI readbacks may differ. Orientation, exposure (`AcquireTime_RBV`: 0.1 ms in training, 1 ms on 10-30), gain or resolution (`RESOLUTION`: 30.5 µm in training, 31.22 on 10-30) differing is a finding to understand before S3: the code follows the live orientation, but the analysis constants were set with the training values;
- the camera ROI didn't change during the 3 s: the readbacks are read once, before the capture, and a move that keeps its size wouldn't be caught. Ask the operator, or run the cell twice and compare `camera`;
- the figure's left panel shows the beam where the ops DTOTR2 display shows it, with the same handedness (compare with the display, or a screenshot of it);
- the beam lies inside the outlined hand crop.

In [ ]:
from twobunch_deploy import session

geometry = session.geometry_test(seconds=3)
geometry

## S3: background and processed frames

Reads the same PVs as S2; adds none. Each frame's hand crop goes through the deployed analysis's processing (`screen.process`, Yiheng Ye's chain with the settings of `xtcav_real_features.py`): subtract a background, replace each pixel above 1000 counts by the median of its 3 × 3 neighbourhood, then set the pixels below 35 counts to 0. Then the training's energy cut (`screen.valley`: until S4 and S5, each frame's own valley) and band statistics (`screen.band_stats`), with x scaled by the live `RESOLUTION`.

Two backgrounds, both subtracted pixel by pixel from the hand crop:
- `take_background()`: the mean of 5 s of frames with no beam on DTOTR2.
- `pedestal()`: one number, the median of every hand-crop pixel over 5 s of frames with beam on, for sessions that start with beam.

Each call saves its record under `data/<date>_S3/` (JSON, `.npz`, figure). A later session can reuse a saved background with `session.load_background(path)`, `path` being the background's `.npz` as its `path` shows it.

**Parasitic:** nothing here asks ops for anything. Take the background whenever DTOTR2 shows no beam spot (the ops display, or S2's figure), and run the rest on whatever beam the machine has. The checks that need the two-bunch beam with the TCAV on (a cut on most frames, the band statistics, `peek()`'s cut and centroids) are in the two-bunch runbook (plan §10), not here.

### Background: no beam on DTOTR2

Pass:
- `frames` about 50;
- every `frame_max` near the dark noise (the 10-30 dark frames peaked at 78–80 counts); a frame with beam reaches hundreds of counts or more. The figure's right panel is the brightest frame and must show no beam spot;
- `background.mean` near the dark level (31.3 counts on 10-30);
- `camera` recorded: the exposure, gain and ROI the background was taken at.

In [ ]:
from twobunch_deploy import session

background = session.take_background(seconds=5)
background.record

### Pedestal: beam on DTOTR2, any configuration

Pass:
- `background.mean` (the pedestal) within about 2 counts of the background's `background.mean` above. On 10-30 the beam frames gave 32–33 against 31.3: the beam's faint halo raises the median a little;
- `fraction_above_threshold` well under half (7.8% on 10-30's beam, 2.2–2.4% in the DAQ runs). Near half or more, the beam fills too much of the crop for the median to be the dark level. Near 0, there was no beam: run it again when there is.

In [ ]:
pedestal = session.pedestal(seconds=5)
pedestal.record

### Processing test: no beam on DTOTR2

Every frame of a 10 s capture, processed once with each background; the `.npz` keeps every frame's hand crop for offline checks.

Pass:
- `frames` about 100;
- `intensity_median` near 0 with both backgrounds: the threshold removes the dark noise. On the 10-30 dark frames: 0 with a live background, about 180 counts per frame with a pedestal; a beam frame sums to 5–8e5;
- `processing_s`: `median` and `max` well under 0.1 s, one frame at 10 Hz.

In [ ]:
processing_off = session.processing_test([background, pedestal], seconds=10)
processing_off

### Processing test: beam on DTOTR2, any configuration

Pass:
- `processing_s`: `median` and `max` well under 0.1 s;
- the two backgrounds' `intensity_median` agree within about 2%. On 10-30's beam, the pedestal gave 0.3–1.5% more than a live background;
- recorded, not judged here: `intensity_median` (DAQ runs 4.7–4.9e5 per frame, 10-30's beam about 8e5), `cut_found` and the band-statistics `median`s, which mean something only on the two-bunch beam.

After copying the records back, the saved crops give a frame-by-frame comparison of the two backgrounds offline (10-30: the processed beam's centroid moved by under 0.1 px, at most 0.5 px).

In [ ]:
processing_on = session.processing_test([background, pedestal], seconds=10)
processing_on

### peek(): any beam

The newest frame through the processing, with the background.

Pass:
- the figure renders: the hand crop, the processed frame and its y projection. A cut appears only where the frame shows a valley; on one bunch or with the TCAV off there may be none;
- `frames` 2 or more and `age_s` under about 1 s: the frame is current;
- `processing_s` recorded.

In [ ]:
session.peek(background)

## S4: TCAV readbacks and tags

Adds two PVs: the XTCAV's amplitude and phase readbacks (`pvs.TCAV_AMPLITUDE`, MV, and `pvs.TCAV_PHASE`, deg; the DAQ's names, checked by hand on srv01 on 11-02: 30 Hz, timestamped with the pulse). Each capture subscribes to both alongside the camera (`acquire.capture`), and each frame takes the readbacks with its pulse ID. Each frame is tagged as Yiheng Ye's `classify_tcav_shots` does (the deployed analysis):
- `+` and `-`: amplitude within 0.25 MV of the mean of the amplitudes above 10 MV, and phase within 1° of +90 and −90;
- `off`: amplitude more than 0.5 MV below the mean amplitude of the `+` and `-` frames (with the TCAV off throughout, below 9.5 MV);
- `neither`: any other readback;
- `unmatched`: no readback with the frame's pulse ID.

The readbacks don't show the toggler's off windows as off: with the klystron on standby, they measure the standby pulse (amplitude near 22 MV, phase slewing to the next polarity) while the beam is unstreaked. Those frames are mostly `neither`; `off` catches the amplitude dip at the trigger switch, a few frames per off window. `step()` records each frame's beam x RMS, which shows whether its off frames were unstreaked.

The `+` and `-` frames less than 1 s after their polarity window began are dropped: the screen keeps settling for about 1 s after the phase locks. A window begins when the polarity changes, after an off frame, or after 5 or more `neither` frames. Records go under `data/<date>_S4/`.

### tcav_test(): any beam, whatever state the TCAV is in

10 s of frames with the readbacks; the `.npz` keeps every frame's pulse ID, readbacks and tag, and every update.

Pass:
- no `TimeoutError`: both PVs answer;
- `updates`: each PV's `count` about three per frame, its `pulse_id_steps` 12 (30 Hz, as read by hand on 11-02). Far fewer updates mean frames go unmatched, and the matching needs another design before a TCAV window;
- `matched_fraction` near 1;
- `tags` as the TCAV is: off → all `off` (amplitude near 0; 11-02 by hand: 0.015–0.02 MV); on at one polarity → `+` or `-` (`range`: amplitude near 22 MV, phase within 1° of ±90); toggling → all four tags;
- `range` in the units expected: MV and degrees.

In a TCAV window (else at the start of two-bunch shift 1): the tags separate off, + and −; frames just after each flip whose image disagrees with its tag set `drop_s`.

In [ ]:
tcav = session.tcav_test(seconds=10)
tcav

### peek(): the newest frame's tag

Pass: the title shows the newest frame's TCAV tag, amplitude and phase, as the TCAV is. The cut is still the frame's own valley, without row correction (`step()` has both).

In [ ]:
session.peek(background)

## S5: row correction and step readings

Adds no PV. `step()` captures for `seconds` with the TCAV readbacks, then reads the frames as the deployed analysis reads a DAQ step (`screen.read_step`; the v7 replay):
1. each `+`, `-` and `off` frame through the S3 processing, with the background;
2. the row-correction reference from the step's off frames: their mean, a Gaussian fitted in x to each row with signal, and each row's shift to the crop's centre; each `+` and `-` frame's rows shifted by it;
3. the step's energy cut: the median of the kept `+` frames' own valleys; each `+` and `-` frame's band statistics at it;
4. the reading from the kept frames: spacing (half-difference of the drive–witness offsets over the streak), drive σz, drive charge fraction, edge fraction, cut sensitivity.

The reading is valid only if every check passes: at least 15 kept frames with statistics per polarity, an off frame (no reference otherwise), a cut, |spacing| above 60 µm, an edge fraction below 0.01, and beam (each polarity's kept frames sum to a median of at least 1e5 counts; beam frames sum to 5–9e5). Everything the frames allow is computed and recorded either way.

`streak_um_per_deg` is the TCAV calibration in µm on the screen per degree, at the live pixel size (`RESOLUTION`, 31.22 µm on 10-30): a calibration measured at another pixel size must be rescaled first (the DAQ runs' 1050–1113 µm/deg are at 30.5). On parasitic beam any value does: 1136 is the DAQ's 1110 rescaled to 31.22.

Records go under `data/<date>_S5/`: the reading, its checks, the counts per tag, the reference fit, timing, the session's peak memory, and each frame's pulse ID, readbacks, tag, drop and band statistics (JSON); every frame's hand crop with the same, the reference and the two median frames (`.npz`). `r.show()` draws the step and saves the figure next to them.

### step(): any beam

Pass:
- it returns. With the TCAV off, `valid` is False and `checks` say why: no `+` or `-` frames, no cut, no beam check;
- with beam and the TCAV off, every frame is `off` and the reference is built from all of them: `reference.rows_fitted` above 0, `shift_columns` and `off_x_rms_mm` recorded (unstreaked: about 0.2 mm);
- `timing_s`: `capture` about `seconds`; `reading` well under the step period (1.5–2 s for about 70 frames on the DAQ steps);
- `peak_memory_mb` recorded (frames at full sensor are 2.5 MB each);
- `r.show()` renders: the readback panels show each frame's tag.

In a TCAV window (else at the start of two-bunch shift 1): steps with both polarities and at least 15 kept frames each, the reference's stability from step to step, the timing cycle. Two-bunch runbook: 20–30 steps with the machine left alone.

In [ ]:
r = session.step(background, streak_um_per_deg=1136, seconds=15)
{k: r.record[k] for k in ("valid", "checks", "tags", "reference", "cut_row", "reading", "timing_s", "peak_memory_mb")}

In [ ]:
r.show()

## S6, S7: L1 and L2 phase PVs

Adds 16 PVs, checked by hand on srv01 first (`to_srv01/phase_pv_check.py`). L1 (`pvs.L1`), the LI11 klystrons 11 and 21: the phase and amplitude readbacks `FAST_PACT` and `FAST_AACT` (the DAQ's BSA names), the phase setpoints `PREQ` (in no DAQ) and the subbooster phase setpoints `SSSB_PDES`. L2 (`pvs.L2`): the subbooster phase setpoints `PDES` of sectors 11–14 and their readbacks `PHAS` (the DAQ's nonBSA names). The L2 multiknob moves LI12–14's 1:1; LI11's phases L2's klystrons 4–8 (F2_live_model), and the multiknob doesn't move it. Read over Channel Access only; if the subbooster PVs don't answer, the AIDA-PVA route is decided then.

All 16 are monitored with the TCAV readbacks during every capture (`step()`, `tcav_test()`, `peek()`), so those calls now stop with a `TimeoutError` naming any of them that doesn't answer. `step()` records each PV's number of updates, median, first and last value and exact zeros (`acquire.summarise`) under `phases`; its `.npz` keeps every update. Nothing is checked at run time: a setpoint that changed during a step shows as first ≠ last.

### phases_test(): any beam, or none

60 s of updates of the 16 PVs, without the camera. The `.npz` keeps every update; the figure shows each PV against time.

Pass:
- no `TimeoutError`: all 16 answer over Channel Access;
- each `FAST_PACT`: `updates` at the rate the by-hand check showed, with regular `pulse_id_steps` if it is timestamped with the pulse; `median` near the DAQ's (TEST_15668 step medians: −24.4 to −23.4°); `std` about 0.13–0.25° (the DAQ's per-step scatter, zeros left out); `zeros` a small fraction of `updates` (DAQ: 0.6–0.8%);
- the two `FAST_PACT` medians within about 0.1° of each other (a feedback moves them together; DAQ steps: up to 0.06°);
- `readback - setpoint` (`FAST_PACT` − `PREQ`) recorded for each station: v2 moves `PREQ` and reads `FAST_PACT`;
- `FAST_AACT` recorded, with its units (DAQ: about 0.50 and 0.55);
- `SSSB_PDES` −18.03 (the DAQ's);
- the LI12–14 subbooster `PDES` equal, and as the multiknob display shows them; LI11's as in the DAQ (0); each `PHAS` within about 1–2° of its `PDES` (TEST_15668, every shot: within 0.6°, 0.8°, 0.7° and 1.7° for LI11, 12, 13, 14, moves of the scan included);
- the setpoints (`PREQ`, `SSSB_PDES`, `PDES`): one update each, unless something moved them.

If ops move L1 or L2 during the session: `FAST_PACT` follows `PREQ`, and the LI12–14 `PDES` move together.

In [ ]:
phases = session.phases_test(seconds=60)
phases

### step(): the L1 and L2 PVs over a step

Uses `background` from S3. Pass:
- `phases` holds all 16 PVs; each `FAST_PACT` has about as many updates per second as in `phases_test()`, and a median close to its median there if nothing moved;
- `r.show()` renders: the bottom panel shows both `FAST_PACT` readbacks during the step with their medians, and its title the `PREQ`s and the L2 PVs.

In [ ]:
r = session.step(background, streak_um_per_deg=1136, seconds=15)
r.record["phases"]

In [ ]:
r.show()